### 1. Обзор структуры ocr_dataset.csv

В этом шаге загружается файл `data/ocr_rvl_cdip/ocr_dataset.csv`, чтобы понять его структуру: список колонок, размер датасета и несколько первых строк. Это нужно для дальнейшего проектирования текстового ML-пайплайна после OCR.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
csv_path = project_root / "data" / "ocr_rvl_cdip" / "ocr_dataset.csv"

print(f"CSV path: {csv_path}")
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))

print("\nHead:")
display(df.head())

print("\nLabel value counts (top 20):")
display(df["label"].value_counts().head(20))

CSV path: /home/dovzhuk/projects/document-checker-course/data/ocr_rvl_cdip/ocr_dataset.csv
Shape: (33014, 4)

Columns: ['path', 'label', 'reference', 'text']

Head:


,path,label,reference,text
0,images_tif/advertisement/0000001863.tif,advertisement,advertisement/0000001863,IMAGE\nNOT\nAVAILABLE\nONLINE\n\nThe material ...
1,images_tif/budget/0000009955.tif,budget,budget/0000009955,Received\nAPR 9 1984\nDr. I. W. Hughes\n\nSTAT...
2,images_tif/email/0001451592.tif,email,email/0001451592,B&W BROWN AND WILLIAMSON TOBACCO CORPORATION\n...
3,images_tif/file_folder/0000011115.tif,file_folder,file_folder/0000011115,Attachment B\n517007520
4,images_tif/form/0000953276.tif,form,form/0000953276,0158\n61 55 40 30\n35 mm\n25 to 40\nThermal Ci...



Label value counts (top 20):


label
memo                      2414
email                     2396
letter                    2345
file_folder               2338
presentation              2327
handwritten               2274
resume                    2242
form                      2215
advertisement             2175
scientific_report         2175
specification             2079
questionnaire             2023
invoice                   2014
budget                    1631
news_article              1331
scientific_publication    1035
Name: count, dtype: int64

### Вывод по пункту 1

Файл `data/ocr_rvl_cdip/ocr_dataset.csv` содержит 33014 строк и 4 колонки: `path`, `label`, `reference`, `text`. Колонка `text` хранит результат OCR по каждому изображению из `images_tif`, а колонка `label` задаёт тип документа (16 классов, таких как `file_folder`, `email`, `memo`, `budget`, `advertisement` и т.д.). Баланс классов относительно ровный: самый частый класс (`memo`) содержит около 2414 примеров, самый редкий (`scientific_publication`) около 1035, что позволяет начать с простых линейных моделей без сложной балансировки.

### 2. Разделение на train/val и базовая фильтрация

На этом шаге отфильтруем строки с пустым или слишком коротким текстом и разделим датасет на обучающую и валидационную части. Это подготовит данные для обучения базовой текстовой модели классификации типа документа.

In [2]:
from sklearn.model_selection import train_test_split

# Копия исходного датафрейма, чтобы не портить df
df_clean = df.copy()

# Убираем строки с пустым или NaN-текстом
df_clean["text"] = df_clean["text"].astype(str).str.strip()
before = len(df_clean)
df_clean = df_clean[df_clean["text"].str.len() > 0]
after = len(df_clean)

print(f"Removed empty texts: {before - after} rows")

# Приводим к обычным Python-спискам
X = df_clean["text"].astype(str).tolist()
y = df_clean["label"].astype(str).tolist()

# Разделение на train / val
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

Removed empty texts: 5 rows
Train size: 26407, Val size: 6602


### Вывод по пункту 2

После фильтрации пустых OCR-текстов из датасета было удалено всего 5 строк, что не влияет существенно на объём данных. Оставшиеся примеры (33009 строк) были разделены на обучающую и валидационную части с долей 80/20 и стратификацией по метке `label`: 26407 документа в train и 6602 в val. Это обеспечивает репрезентативное покрытие всех 16 классов как на обучении, так и на валидации.

### 3. Базовая модель: TF‑IDF + Logistic Regression

На этом шаге строим базовый текстовый пайплайн: TF‑IDF-векторизатор по полю `text` и линейный классификатор (Logistic Regression). Модель обучим на `X_train, y_train` и оценим на `X_val, y_val` по метрикам accuracy и macro F1.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Базовый TF-IDF + Logistic Regression пайплайн
text_clf = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            lowercase=True,
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
            class_weight=None,
        )),
    ]
)

# Обучение
text_clf.fit(X_train, y_train)

# Предсказание на валидации
y_val_pred = text_clf.predict(X_val)

acc = accuracy_score(y_val, y_val_pred)
f1_macro = f1_score(y_val, y_val_pred, average="macro")

print(f"Validation accuracy: {acc:.4f}")
print(f"Validation macro F1: {f1_macro:.4f}")

print("\nClassification report:")
print(classification_report(y_val, y_val_pred))

/home/dovzhuk/projects/document-checker-course/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Validation accuracy: 0.8135
Validation macro F1: 0.8132

Classification report:
                        precision    recall  f1-score   support

         advertisement       0.87      0.82      0.84       435
                budget       0.86      0.85      0.86       326
                 email       0.95      0.90      0.92       479
           file_folder       0.63      0.92      0.75       467
                  form       0.79      0.76      0.78       443
           handwritten       0.84      0.72      0.78       455
               invoice       0.93      0.91      0.92       403
                letter       0.78      0.74      0.76       469
                  memo       0.78      0.80      0.79       483
          news_article       0.72      0.77      0.74       266
          presentation       0.69      0.65      0.67       465
         questionnaire       0.94      0.80      0.87       405
                resume       0.97      0.95      0.96       448
scientific_publication 

### Вывод по пункту 3

Базовый текстовый пайплайн TF‑IDF + Logistic Regression на текущем корпусе даёт на валидации качество около `0.81` по accuracy и `0.81` по macro F1 (`accuracy = 0.8135`, `macro F1 = 0.8132`). Для задачи классификации 16 типов документов по OCR‑тексту это уже уверенный стартовый бейзлайн.

Модель особенно хорошо распознаёт классы с устойчивой структурой текста и характерной лексикой, такие как `resume`, `email`, `invoice`, `questionnaire`, `specification` и частично `advertisement`, где F1‑score находится на уровне `0.84–0.96`. Сложнее всего даются более разнообразные и менее формализованные классы (`file_folder`, `presentation`, `news_article`, `scientific_report`), где тексты сильно различаются по содержанию и стилю, из‑за чего качество заметно ниже.

В целом полученные метрики показывают, что комбинация TF‑IDF и Logistic Regression хорошо подходит для базовой текстовой классификации в условиях OCR‑шума. Этот результат можно использовать как отправную точку: дальше для улучшения качества имеет смысл пробовать очистку текста, тонкую настройку параметров TF‑IDF и альтернативные линейные модели.

### 4. Сохранение обученного пайплайна

На этом шаге сохраняем обученный пайплайн `TF‑IDF + Logistic Regression` в каталог `artifacts/models/` с помощью `joblib`. Это позволит в дальнейшем загружать модель без повторного обучения и использовать её в основном приложении и на демо.

In [4]:
from pathlib import Path
import joblib

project_root = Path.cwd().parent
models_dir = project_root / "artifacts" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "text_tfidf_logreg.joblib"

joblib.dump(text_clf, model_path)

print(f"Model saved to: {model_path}")

Model saved to: /home/dovzhuk/projects/document-checker-course/artifacts/models/text_tfidf_logreg.joblib


### Вывод по пункту 4

Обученный пайплайн `TF‑IDF + Logistic Regression` сохранён в виде артефакта по пути `artifacts/models/text_tfidf_logreg.joblib`. Теперь модель можно загружать без повторного обучения и использовать в основном коде и демо-сценариях (`document → OCR → text → ML → result`). Это закрывает базовое требование курсового проекта по наличию сохранённой воспроизводимой текстовой модели.

### 5. Лёгкая очистка текста и модель Linear SVM

На этом шаге добавляем простую предобработку OCR-текста (нормализация регистра и пробелов) и обучаем альтернативную модель на тех же данных: TF‑IDF + LinearSVC (Linear SVM). Цель — проверить, удастся ли улучшить качество по сравнению с бейзлайном TF‑IDF + Logistic Regression.

In [5]:
import re
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

class SimpleTextCleaner(BaseEstimator, TransformerMixin):
    """Простая очистка текста: lower, убираем лишние пробелы и управляющие символы."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cleaned = []
        for text in X:
            t = str(text)
            t = t.lower()
            # Убираем управляющие символы и лишние пробелы
            t = re.sub(r"\s+", " ", t)
            t = t.strip()
            cleaned.append(t)
        return cleaned

# Пайплайн: очистка текста -> TF-IDF -> Linear SVM
svm_clf = Pipeline(
    steps=[
        ("clean", SimpleTextCleaner()),
        ("tfidf", TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            lowercase=False,  # lowercase уже делаем в cleaner
        )),
        ("clf", LinearSVC())
    ]
)

# Обучение
svm_clf.fit(X_train, y_train)

# Предсказание на валидации
y_val_pred_svm = svm_clf.predict(X_val)

acc_svm = accuracy_score(y_val, y_val_pred_svm)
f1_macro_svm = f1_score(y_val, y_val_pred_svm, average="macro")

print(f"Validation accuracy (LinearSVC): {acc_svm:.4f}")
print(f"Validation macro F1 (LinearSVC): {f1_macro_svm:.4f}")

print("\nClassification report (LinearSVC):")
print(classification_report(y_val, y_val_pred_svm))

Validation accuracy (LinearSVC): 0.8376
Validation macro F1 (LinearSVC): 0.8361

Classification report (LinearSVC):
                        precision    recall  f1-score   support

         advertisement       0.85      0.86      0.86       435
                budget       0.85      0.86      0.85       326
                 email       0.95      0.92      0.94       479
           file_folder       0.71      0.88      0.79       467
                  form       0.81      0.77      0.79       443
           handwritten       0.87      0.75      0.81       455
               invoice       0.93      0.92      0.92       403
                letter       0.81      0.78      0.79       469
                  memo       0.84      0.84      0.84       483
          news_article       0.77      0.84      0.80       266
          presentation       0.72      0.68      0.70       465
         questionnaire       0.90      0.86      0.88       405
                resume       0.97      0.98      0.

### Вывод по пункту 5

Модель TF‑IDF + LinearSVC с простой очисткой текста показывает на текущем корпусе высокое качество: `accuracy = 0.8376`, `macro F1 = 0.8361`. Для задачи классификации 16 типов документов по OCR‑тексту это сильный результат, который подтверждает, что линейная SVM хорошо подходит для данной постановки.

Лучше всего модель распознаёт классы с наиболее характерной структурой и словарём, такие как `resume`, `email`, `invoice`, `specification`, `questionnaire` и `advertisement`, где F1-score находится примерно в диапазоне `0.86–0.97`. Более сложными остаются классы с менее устойчивым содержанием и более шумным OCR, например `presentation`, `scientific_report`, `scientific_publication`, `file_folder` и частично `letter`, однако и по ним качество остаётся на приемлемом уровне.

В целом полученные метрики показывают, что пайплайн `SimpleTextCleaner → TF‑IDF → LinearSVC` является сильной основной текстовой моделью для текущего этапа проекта. Именно эту модель разумно использовать как основной кандидат для интеграции в итоговый пайплайн `document → OCR → text → ML → result`.

### 6. Сохранение улучшенной модели LinearSVC

После того как модель TF‑IDF + LinearSVC показала лучшее качество на валидации, на этом шаге сохраняем её в каталог `artifacts/models/` как отдельный артефакт. Это позволит использовать LinearSVC в основном приложении и при демонстрации без повторного обучения, а также сравнивать её с базовой моделью Logistic Regression.

In [6]:
from pathlib import Path
import joblib

project_root = Path.cwd().parent
models_dir = project_root / "artifacts" / "models"
models_dir.mkdir(parents=True, exist_ok=True)

svm_model_path = models_dir / "text_tfidf_linearsvc.joblib"

joblib.dump(svm_clf, svm_model_path)

print(f"SVM model saved to: {svm_model_path}")

SVM model saved to: /home/dovzhuk/projects/document-checker-course/artifacts/models/text_tfidf_linearsvc.joblib


### Вывод по шагу 6

Улучшенный пайплайн `SimpleTextCleaner → TF‑IDF → LinearSVC` сохранён как артефакт по пути `artifacts/models/text_tfidf_linearsvc.joblib`. Теперь в проекте есть два варианта текстовой модели: базовый (Logistic Regression) и улучшенный (LinearSVC). Для демонстрации и интеграции в общий пайплайн разумно использовать именно LinearSVC как текущую лучшую модель по валидационным метрикам.